# 🛡️ Adaptive Temporal-Relational IDS: Dataset Ingestion & Metadata Pipeline

**Project:** *Adaptive Temporal-Relational Intrusion Detection for Evolving Network Traffic: Investigating Classical, Quantum, and Continual Learning Approaches*

This Google Colab notebook provides an automated, one-click environment to:
1. Mount Google Drive to persist downloaded datasets and metadata without consuming local laptop resources.
2. Download candidate datasets (`NF-CSE-CIC-IDS2018-v2`, `NF-UNSW-NB15-v2`, `NF-ToN-IoT-v2`, `CSE-CIC-IDS2018`, `CICIoT2023`, etc.).
3. Run automated schema profiling and metadata extraction to evaluate **Temporal & Relational readiness** ($G_t = (V_t, E_t)$ graph node/edge identifiers, microsecond timestamps, IAT features, label distributions).

## Step 1: Environment Setup & Google Drive Mounting

In [ ]:
# Mount Google Drive
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except ImportError:
    print("[INFO] Running outside Colab environment.")

# Set up working directories
PROJECT_DIR = Path("/content/drive/MyDrive/IDS_Research_Project")
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
METADATA_DIR = PROJECT_DIR / "metadata_reports"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw Data Path: {RAW_DATA_DIR}")
print(f"Metadata Reports Path: {METADATA_DIR}")

## Step 2: Install Pipeline Dependencies

In [ ]:
# Install lightweight ingestion dependencies
!pip install -q pyyaml kaggle kagglehub polars pyarrow tabulate rich
print("[SUCCESS] Dependencies installed.")

## Step 3: Configure Kaggle API (Optional / Recommended for Kaggle Datasets)
If downloading datasets hosted on Kaggle (e.g. `NF-CSE-CIC-IDS2018-v2`, `NF-UNSW-NB15-v2`), upload your `kaggle.json` or set your API credentials below.

In [ ]:
import os

# OPTION A: Set environment variables directly
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_api_key'

# OPTION B: Upload kaggle.json file directly to Colab runtime
from google.colab import files
import shutil

kaggle_json_path = Path("/root/.kaggle/kaggle.json")
if not kaggle_json_path.exists():
    print("Upload your kaggle.json (if using Kaggle-hosted datasets):")
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move('kaggle.json', str(kaggle_json_path))
        os.chmod(str(kaggle_json_path), 0o600)
        print("[SUCCESS] kaggle.json configured.")

## Step 4: Run Dataset Downloader
Download Tier 1 Core Primary datasets (`NF-CSE-CIC-IDS2018-v2` & `NF-UNSW-NB15-v2`) or Tier 2 Cross-Domain datasets.

In [ ]:
# Run download script for Tier 1 (Core Primary Standardized NetFlow Suite)
!python scripts/download_datasets.py --tier 1 --output-dir /content/drive/MyDrive/IDS_Research_Project/data/raw

## Step 5: Extract Metadata and Assess Temporal-Relational Readiness

In [ ]:
# Execute metadata extraction & profile generation
!python scripts/extract_metadata.py \
    --data-dir /content/drive/MyDrive/IDS_Research_Project/data/raw \
    --output-dir /content/drive/MyDrive/IDS_Research_Project/metadata_reports

## Step 6: Interactive Metadata Report Preview

In [ ]:
from IPython.display import Markdown, display

report_files = list(METADATA_DIR.glob("*_metadata.md"))
print(f"Found {len(report_files)} metadata reports:")
for r in report_files:
    print(f" - {r.name}")

if report_files:
    # Display the first report
    with open(report_files[0], 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))